# **Project Title: NASDAQ Financials & Fundamentals (2024 - 2026)**

**Project Goal:** To analyze the financial performance and fundamentals of **NASDAQ**-listed companies from 2024 to 2026 using financial statement data,and identify trends,patterns,and key factors affecting company performance.

Data set Information



*  Source :  Kaggle
*  Total Rows:  22,785
*  Total Columns: 28
*  Date Range: January 2024 to August 2026


















In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import requests

In [ ]:
url = "https://www.kaggle.com/datasets/abdurrahmanhakanacar/nasdaq-financials-and-fundamentals-2024-2026"

In [ ]:
!pip install kagglehub --quiet
import kagglehub
import pandas as pd
import os

# Download the latest version of the dataset (you will need to enter your Kaggle credentials if prompted)
path = kagglehub.dataset_download("abdurrahmanhakanacar/nasdaq-financials-and-fundamentals-2024-2026")

print("Dataset downloaded to:", path)

# Find the CSV file inside the downloaded directory
files = os.listdir(path)
csv_files = [f for f in files if f.endswith('.csv')]

if csv_files:
    df = pd.read_csv(os.path.join(path, csv_files[0]))
    print("Successfully loaded dataset! Shape:", df.shape)
else:
    print("No CSV file found in the downloaded files:", files)

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.columns.tolist()

# **Check Missing Values**

Missing values identifed in this dataset they are listed below:

---



fiscal_year - 1 , fiscal_period -1 ,revenues - 4754 ,cost_of_revenue -11090 , gross_profit - 11278, rd_expense - 10810 ,sga_expense -3566 ,operating_income -3665 ,cash_and_equivalents - 848 ,current_assets -3563 ,total_assets - 57 , current_liabilities -3658,long_term_debt - 13324,total_liabilities - 2085 ,stockholders_equity - 129 ,operating_cash_flow - 278 ,capital_expenditures-5276 free_cash_flow-5408,eps_basic -1754,eps_diluted- 2267,shares_outstanding- 1123




In [ ]:
df.isnull().sum()


In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.columns[df.isnull().all()]

In [ ]:
# See % missing
missing = df.isnull().mean().sort_values(ascending=False) * 100
print(missing[missing > 0])

# If a column has >60% missing, better to drop it
cols_to_drop = missing[missing > 60].index
df = df.drop(columns=cols_to_drop)

In [ ]:
# 1. For critical financial metrics - Groupwise Median (by ticker)
# Use actual column names matching your NASDAQ dataset (snake_case)
for col in ['cost_of_revenue', 'long_term_debt', 'revenues', 'net_income']:
    if col in df.columns:
        df[col] = df.groupby('ticker')[col].transform(
            lambda x: x.fillna(x.median()) if not x.median() is np.nan else x
        )

# 2. For price/volume - Forward Fill (time series logic)
# Note: This dataset does not have daily stock prices like 'open'/'close', but if they are added later:
price_cols = ['open', 'close', 'high', 'low', 'volume']
for col in price_cols:
    if col in df.columns:
        df[col] = df.groupby('ticker')[col].ffill()

# 3. For ratios/fundamentals - Industry/Sector median is better if 'sector' column exists
if 'sector' in df.columns:
    for col in ['peRatio', 'pbRatio', 'debtToEquity']:
        if col in df.columns:
            df[col] = df.groupby('sector')[col].transform(
                lambda x: x.fillna(x.median())
            )

# 4. If still missing, then use overall median
df = df.fillna(df.median(numeric_only=True))

# 5. For target column - DON'T impute, DROP ROWS

target_col = 'net_income'
if target_col in df.columns:
    df = df.dropna(subset=[target_col])

# Detect duplicate records

In [ ]:
df = df.dropna(axis=1, how="all")


In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)]

# Remove exact duplicate rows

In [ ]:
import os
# Restore 'df' from the cached CSV path
csv_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_path)

# Safely execute the duplication removal
df = df.drop_duplicates(keep="first")

In [ ]:
# Count duplicate rows
print("Total duplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows only if appropriate
df = df.drop_duplicates()

print("Dataset shape after duplicate check:", df.shape)

In [ ]:
df.duplicated().sum()

# Fix inconsistent values

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()


In [ ]:
key_columns = ["ticker", "period_end"]

if all(col in df.columns for col in key_columns):
    duplicate_keys = df[
        df.duplicated(subset=key_columns, keep=False)
    ]

In [ ]:
numeric = df.select_dtypes(include=np.number)
numeric = numeric.loc[:, numeric.nunique(dropna=True) > 2]


In [ ]:
ticker_cols = [
    col for col in df.columns
    if col.lower() in ["ticker", "symbol", "stock_ticker"]
]

for col in ticker_cols:
    df[col] = df[col].str.upper()


In [ ]:
category_names = [
    "sector", "industry", "exchange",
    "currency", "country", "category"
]

for col in df.columns:
    if col.lower() in category_names:
        df[col] = (
            df[col].astype("string").str.strip().str.title()
        )


Detect outliers using the IQR method

---
The Interquartile Range (IQR) method identifies numerical values that fall unusually far from the middle 50% of a column.

IQR=Q3−Q1

Lower Bound=Q1−1.5(IQR)

Upper Bound=Q3+1.5(IQR)


In [ ]:
results = []

for col in numeric.columns:
    s = numeric[col].dropna()
    if len(s) == 0:
        continue

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (df[col] < lower) | (df[col] > upper)

    results.append({
        "Column": col,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(mask.sum()),
        "Outlier_Percent": round(mask.mean() * 100, 2)
    })

In [ ]:
col = "shares_outstanding"

display(
    df[col].describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# Display the largest values for investigation
display(df.nlargest(10, col)[[col]])

In [ ]:
summary = pd.DataFrame(results)
summary.sort_values(
        "Outlier_Count", ascending=False
    )

# Check numerical columns and invalid values

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()


In [ ]:
icker_cols = [
    col for col in df.columns
    if col.lower() in ["ticker", "symbol", "stock_ticker"]
]

for col in ticker_cols:
    df[col] = df[col].str.upper()


In [ ]:
category_names = [
    "sector", "industry", "exchange",
    "currency", "country", "category"
]

for col in df.columns:
    if col.lower() in category_names:
        df[col] = (
            df[col].astype("string").str.strip().str.title()
        )


# Standardize categories

In [ ]:
for col in text_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(20))

In [ ]:
category_mapping = {
    "Tech": "Technology",
    "Information Technology": "Technology"
}

for col in df.columns:
    if col.lower() in ["sector", "industry"]:
        df[col] = df[col].replace(category_mapping)

# Check and validate date columns

In [ ]:
date_cols = [
    col for col in df.columns
    if any(word in col.lower()
           for word in ["date", "time", "period"])
]


In [ ]:
for col in date_cols:
    original = df[col].copy()
    parsed = pd.to_datetime(original, errors="coerce")

In [ ]:
invalid_mask = original.notna() & parsed.isna()

In [ ]:
if invalid_mask.any():
    print("Examples of invalid values:")
    print(original[invalid_mask].head(10).tolist())

df[col] = parsed

print("Minimum date:", df[col].min())
print("Maximum date:", df[col].max())

In [ ]:
for col in date_cols:
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        outside_range = (
            df[col].notna()
            & ~df[col].between(
                pd.Timestamp("2024-01-01"),
                pd.Timestamp("2026-08-31")
            )
        )

In [ ]:
print(
            f"{col} dates outside expected range:",
            outside_range.sum()
        )

In [ ]:
# Define numeric_cols globally from the DataFrame
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    print(f"\nColumn: {col}")
    print("Negative values:", (df[col] < 0).sum())
    print("Zero values:", (df[col] == 0).sum())
    print("Infinite values:", np.isinf(df[col]).sum())

In [ ]:
df[numeric_cols] = df[numeric_cols].replace(
    [np.inf, -np.inf], np.nan
)

# Advanced method -Use KNN iterate imputer

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer

# Select only numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
X = df[numeric_cols]

# KNN - finds similar companies and fills
imputer = KNNImputer(n_neighbors=5)
df[numeric_cols] = imputer.fit_transform(X)

# OR Iterative (MICE) - most accurate for finance
# imputer = IterativeImputer(max_iter=10, random_state=42)
# df[numeric_cols] = imputer.fit_transform(X)

In [ ]:
def derive_fiscal_period(row):
  if pd.notna(row["fiscal_period"]):
    return row["fiscal_period"]

  # If Form is 10-K, it is almost always the full fiscal year (often designated as 'FY' or 'Q4')
  if pd.notna(row.get("form")) and "10-K" in str(row["form"]):
    return "FY"  # or 'Q4' depending on dataset convention

  # Otherwise, infer from the month of the period_end_date
  if pd.notna(row["period_end_date"]):
    month = pd.to_datetime(row["period_end_date"]).month
    # Standard mapping for calendar-year companies:
    # Q1 ends in March (3), Q2 in June (6), Q3 in September (9), Q4/FY in December (12)
    month_to_q = {3: "Q1", 6: "Q2", 9: "Q3", 12: "Q4"}
    return month_to_q.get(month, np.nan)

  return np.nan


# Apply the derivation function
df["fiscal_period"] = df.apply(derive_fiscal_period, axis=1)

In [ ]:

# Final data quality report

quality_report = pd.DataFrame({
    "Check": [
        "Total rows",
        "Total columns",
        "Missing cells",
        "Duplicate rows",
        "Numerical columns",
        "Categorical/text columns",
        "Date columns"
    ],
    "Result": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        len(df.select_dtypes(include=np.number).columns),
        len(df.select_dtypes(include=["object", "string"]).columns),
        len(date_cols)
    ]
})

print(quality_report.to_string(index=False))



# INITIAL STRUCTURE & OVERVIEW (EDA)

In [ ]:
print("\n--- Dataset Shape ---")
print(df.shape)

print("\n--- Columns & Data Types ---")
print(df.dtypes)

print("\n--- Summary Statistics (Numerical Features) ---")
print(
    df[
        [
            "revenues",
            "net_income",
            "operating_cash_flow",
            "free_cash_flow",
            "total_assets",
        ]
    ].describe()
)

# Check missing values percentage
missing_data = (
    df.isnull().sum() / len(df) * 100
).sort_values(ascending=False)
print("\n--- Missing Values Percentage (Top 10 Columns) ---")
print(missing_data.head(10))
#DATA CLEANING & FEATURE ENGINEERING
# ---------------------------------------------------------
# Ensure proper sorting by date
if "period_end_date" in df.columns:
  df["period_end_date"] = pd.to_datetime(df["period_end_date"])
  df = df.sort_values(by=["ticker", "period_end_date"])

# Create financial ratios for screening (handling potential division by zero)
df["net_profit_margin"] = np.where(
    df["revenues"] != 0, df["net_income"] / df["revenues"], np.nan
)
df["fcf_margin"] = np.where(
    df["revenues"] != 0, df["free_cash_flow"] / df["revenues"], np.nan
)

# VISUALIZATIONS

# Horizontal Bar Chart

---
comparison of Top &Bottom 10 Companies to idenify the most profitable and loss making companies


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert date column to datetime
df["period_end_date"] = pd.to_datetime(df["period_end_date"])

# 2. Filter by filing type (e.g., annual '10-K') and period window (2024-2026)
target_filing_type = "10-K"  # Ensure uniform comparison type
filtered_df = df[
    (df["form_type"] == target_filing_type)
    & (df["period_end_date"].dt.year.between(2024, 2026))
]

#  Select the latest comparable period for each company within the window
comparable_df = (
    filtered_df.sort_values("period_end_date")
    .groupby("ticker")
    .last()
    .reset_index()
)

# Sort companies by net income from highest to lowest
sorted_df = comparable_df.sort_values(by="net_income", ascending=False)

# Identify top profitable and most loss-making companies
top_profitable = sorted_df.head(10)
most_loss_making = sorted_df.tail(10).sort_values(
    by="net_income", ascending=True
)

print("--- Top Profitable Companies ---")
print(top_profitable[["ticker", "net_income", "period_end_date"]])

print("\n--- Most Loss-Making Companies ---")
print(most_loss_making[["ticker", "net_income", "period_end_date"]])

# ==========================================
# Chart 1: Top Profitable Companies
# ==========================================
plt.figure(figsize=(11, 6))
bars_top = plt.barh(
    top_profitable["ticker"][::-1],
    top_profitable["net_income"][::-1] / 1e9,
    color="forestgreen",
)
plt.title(
    "Top 10 Most Profitable NASDAQ Companies (2024–2026)",
    fontsize=14,
    pad=15,
)
plt.xlabel("Net Income (USD Billions)", fontsize=12)
plt.ylabel("Ticker Symbol", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)

# Label values in USD billions
for bar in bars_top:
  width = bar.get_width()
  plt.text(
      width + (width * 0.01),
      bar.get_y() + bar.get_height() / 2,
      f"${width:.2f}B",
      va="center",
      ha="right",
      fontsize=10,
      fontweight="bold",
  )

plt.tight_layout()
plt.show()

# ==========================================
#  Chart 2: Most Loss-Making Companies
# ==========================================
plt.figure(figsize=(11, 6))
bars_bottom = plt.barh(
    most_loss_making["ticker"][::-1],
    most_loss_making["net_income"][::-1] / 1e9,
    color="crimson",
)
plt.title(
    "Top 10 Most Loss-Making NASDAQ Companies (2024–2026)",
    fontsize=14,
    pad=15,
)
plt.xlabel("Net Income (USD Billions)", fontsize=12)
plt.ylabel("Ticker Symbol", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)

# Label values in USD billions (handling negative positioning)
for bar in bars_bottom:
  width = bar.get_width()
  ha_pos = "left" if width < 0 else "right"
  offset = -0.5 if width < 0 else 0.2
  plt.text(
      width + offset,
      bar.get_y() + bar.get_height() / 2,
      f"${width:.2f}B",
      va="center",
      ha=ha_pos,
      fontsize=10,
      fontweight="bold",
  )

plt.tight_layout()
plt.show()

### Key Observations

---


*Market Leader:Microsoft(MSFT) Rank#1 as the most Profitable Company in the dataset,generating$133.75 billion in netincome.


*Top Tier ($100B+ Net Income):

   The top 4 companies—MSFT ($133.75B), GOOGL ($132.17B), NVDA ($120.07B), and AAPL ($112.01B)—each generated over $100 billion.  

   Together, these top 4 companies account for the vast majority of total net income among the group.


*Big Tech Dominance:
  
  Tech giants dominate the upper ranking: MSFT, GOOGL, NVDA, AAPL, AMZN ($77.67B), and META ($60.46B) occupy the top 6 spots.



*Significant Profit Drop-off:

  There is a steep drop after META ($60.46B).   

  Walmart (WMT) ranks 7th with $21.89 billion, which is nearly a third of Meta's net income and less than a sixth of Microsoft's.

*Lower Tier Companies:

   Comcast (CMCSA) follow closely at $20.00B.   

   Cisco (CSCO) stands at $13.27B.   

   SNDK rounds out the top 10 at $11.43B.  








# Top Free Cash Flow (FCF) Generators (Horizontal Bar Chart)

In [ ]:


#  Filter for the most recent reporting period
latest_period = df["period_end_date"].max()
latest_df = df[df["period_end_date"] == latest_period].copy()

# Calculate FCF Margin (%) -> Free Cash Flow divided by Revenues
latest_df["fcf_margin"] = (
    latest_df["free_cash_flow"] / latest_df["revenues"]
) * 100

# Drop any rows with missing revenue or FCF data to avoid errors
latest_df = latest_df.dropna(subset=["fcf_margin"])

# Sort by FCF margin to see the highest cash efficiency at the top
# Let's grab the top 15 companies for a comprehensive view
top_fcf_margin = latest_df.sort_values(by="fcf_margin", ascending=False).head(15)

# Define colors: Green for positive margins, Crimson for negative margins
bar_colors = [
    "forestgreen" if val >= 0 else "crimson"
    for val in top_fcf_margin["fcf_margin"]
]

# ==========================================
#  Plotting the Horizontal Bar Chart
# ==========================================
plt.figure(figsize=(12, 7))
bars = plt.barh(
    top_fcf_margin["ticker"][::-1],
    top_fcf_margin["fcf_margin"][::-1],
    color=bar_colors[::-1],
)

plt.title(
    f"Free Cash Flow Margin (FCF / Revenue) Ranking (As of {latest_period})",
    fontsize=14,
    pad=15,
)
plt.xlabel("FCF Margin (%)", fontsize=12)
plt.ylabel("Ticker Symbol", fontsize=12)

# Add a vertical line at 0% to clearly separate positive and negative performance
plt.axvline(0, color="black", linewidth=1, linestyle="-")
plt.grid(axis="x", linestyle="--", alpha=0.6)

# Label values on the bars, dynamically adjusting offset for negative numbers
for bar in bars:
  width = bar.get_width()
  ha_pos = "left" if width < 0 else "right"
  offset = -1.5 if width < 0 else 0.8
  plt.text(
      width + offset,
      bar.get_y() + bar.get_height() / 2,
      f"{width:.1f}%",
      va="center",
      ha=ha_pos,
      fontsize=10,
      fontweight="bold",
  )

plt.tight_layout()
plt.show()

### Key Observations

---

Top Cash Generator:

Photronics, Inc. (PLAB) stands out with the highest positive FCF margin by a wide margin, converting 41.8% of its revenue into free cash flow.   

Positive Cash Flow Companies:

PLAB (+41.8%), Lululemon Athletica (LULU) (+12.9%), and Dialysis Holdings / DLTH (+10.7%) all maintain positive free cash flow margins.  

Among positive generators, PLAB produces over 3x the cash margin of LULU.   

Negative Cash Flow / Cash Burners:

Lovesac (LOVE) shows a modest negative FCF margin of -14.3%, burning relatively little cash compared to its revenue.   

Hooker Furnishings (HOFT) represents an extreme negative outlier, recording a severe FCF margin of -517.7%. This indicates that its cash burn significantly exceeded total revenue generated for the reporting period.  

 Significant Performance Gap:

 There is a 559.5 percentage point spread between the most cash-efficient company (PLAB at +41.8%) and the heaviest cash burner (HOFT at -517.7%) in this group.   


# Distribution of Free Cash Flow Across Sectors/Filings (Hist Plot)

In [ ]:


# 1. Calculate valid observations count
fcf_series = df["free_cash_flow"].dropna() / 1e9
valid_count = fcf_series.count()
print(f"Total number of valid FCF observations: {valid_count}")

# ==========================================
#  Separate Histograms: Positive vs. Negative FCF
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Negative Free Cash Flow (Cash Burners)
sns.histplot(
    fcf_series[fcf_series < 0],
    bins=30,
    kde=True,
    color="crimson",
    ax=axes[0],
)
axes[0].set_title(
    "Distribution of Negative Free Cash Flow (Cash Burn)", fontsize=12
)
axes[0].set_xlabel("Free Cash Flow ($ Billions)", fontsize=10)
axes[0].set_ylabel("Frequency", fontsize=10)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Positive Free Cash Flow (Cash Generators)
sns.histplot(
    fcf_series[fcf_series >= 0],
    bins=50,
    kde=True,
    color="forestgreen",
    ax=axes[1],
)
axes[1].set_title(
    "Distribution of Positive Free Cash Flow (Cash Generators)", fontsize=12
)
axes[1].set_xlabel("Free Cash Flow ($ Billions)", fontsize=10)
axes[1].set_ylabel("Frequency", fontsize=10)
axes[1].set_xlim(0, 50)  # Zoom in on primary positive cluster
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# ==========================================
# Distribution Analysis Grouped by Filing Type
# ==========================================
df_clean = df.copy()
df_clean["fcf_billions"] = df_clean["free_cash_flow"] / 1e9

g = sns.displot(
    data=df_clean.dropna(subset=["free_cash_flow", "form_type"]),
    x="fcf_billions",
    hue="form_type",
    kind="hist",
    bins=50,
    kde=True,
    palette="muted",
    height=5,
    aspect=1.8,
)
g.set(xlim=(-20, 50))
g.fig.suptitle(
    "Free Cash Flow Distribution Across NASDAQ Filings by Filing Type",
    y=1.03,
    fontsize=14,
)
g.set_axis_labels("Free Cash Flow ($ Billions)", "Frequency")
plt.show()

### Key Observations

---


### 1. Free Cash Flow (FCF) Split: Cash Burn vs. Cash Generators

* **Negative FCF (Cash Burn - Left Chart):** The distribution is heavily skewed toward zero. The vast majority of companies experiencing negative cash flow cluster tightly between $0$ and $-\$2\text{ billion}$, indicating that most "burning" firms operate with relatively low cash burn magnitude. A long, thin tail stretches out toward $-\$25\text{ billion}$ for heavily capital-intensive or distressed outliers.
* **Positive FCF (Cash Generators - Right Chart):** The distribution of profitable companies is similarly front-loaded near zero (peaking around the $0\text{ to }3\text{ billion}$ range). The curve drops off steeply, showing that companies generating massive positive free cash flow (stretching past $\$10\text{ to }40\text{ billion}$) are rare outliers.

### 2. Filing Type Breakdown (NASDAQ Filings)

* **Quarterly (10-Q) vs. Annual (10-K) Volume:** The frequency distribution heavily mirrors reporting cycles. Quarterly filings (**10-Q**, shown in blue) vastly outnumber annual reports (**10-K**, shown in orange) by roughly a $4\text{ to }1$ ratio, which aligns with companies filing three quarters and one annual report per fiscal year.
* **Tight Clustering Near Zero:** Across both filing types, the bulk of financial data points compress intensely within the $-\$2\text{ billion}$ to $+\$3\text{ billion}$ band, demonstrating that median company-level cash flows in the dataset hover very close to breakeven.


# Correlation Heatmap (Financial Metrics)

In [ ]:

metrics = [
    "revenues",
    "net_income",
    "operating_cash_flow",
    "free_cash_flow",
    "total_assets",
    "long_term_debt",
]

#  Filter out missing values and get total sample count (N)
corr_df = df[metrics].dropna()
n_obs = len(corr_df)

# Log-scale positive scale metrics (e.g., revenues, total_assets, long_term_debt)
# Note: For metrics with negative values (net_income, FCF, OCF), log-transforming
# requires signed log transforms like np.sign(x) * np.log1p(np.abs(x))
corr_df_transformed = corr_df.copy()
for col in ["revenues", "total_assets", "long_term_debt"]:
    corr_df_transformed[col] = np.log1p(
        corr_df_transformed[col].clip(lower=0)
    )

#  Compute correlation matrix
corr_matrix = corr_df_transformed.corr()

# Plot Heatmap with N in the title
plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
)

plt.title(
    f"Correlation Matrix of NASDAQ Fundamentals (Log-Scaled / Adjusted)\n(N = {n_obs:,} observations)"
)
plt.tight_layout()
plt.show()

### Key Observations

---




### 1. High Cash Flow and Profitability Alignment

* **Operating Cash Flow & Net Income ($0.89$):** There is an exceptionally strong positive association between net income and operating cash flow, indicating that profitable NASDAQ firms tightly couple bottom-line earnings with cash inflows.


* **Operating Cash Flow & Free Cash Flow ($0.88$):** Operating cash flow and free cash flow move closely together, showing that operational cash generation is the primary engine of free cash availability.


* **Net Income & Free Cash Flow ($0.79$):** A strong positive association demonstrates that higher earnings reliably translate to higher free cash flow reserves.



### 2. Scale, Assets, and Leverage

* **Revenues & Total Assets ($0.62$):** A robust positive association exists between top-line revenue and total asset base, reflecting that larger scale and asset deployment generally correspond to higher revenue generation capacity.


* **Total Assets & Long-Term Debt ($0.50$):** A moderate-to-strong positive association indicates that asset-heavy operations across NASDAQ listings often rely correspondingly on long-term debt financing.



### 3. Disconnect Between Top-Line Scale and Margins

* **Revenue vs. Earnings/Cash Flows ($0.12 \text{ to } 0.15$):** Interestingly, after log-scaling and adjustment, the association between `revenues` and profitability metrics (`net_income`, `operating_cash_flow`, `free_cash_flow`) is relatively weak. This highlights diverse business models within NASDAQ where high-revenue firms may experience varying cost structures, high reinvestment rates, or net losses (common among growth and tech companies).



### 4. Methodological Guardrail

* **Association vs. Causation:** These correlation coefficients quantify statistical association across log-scaled financial statements, but **do not imply direct causation**. Sector differences, capital intensity, and differing business life cycles heavily influence these patterns.


# Sector-Wise Profitability Comparison (Box Plot )

In [ ]:

#  Prepare data and calculate Debt-to-Assets ratio
df_leverage = df[["form_type", "long_term_debt", "total_assets"]].dropna()

# Avoid division by zero
df_leverage = df_leverage[df_leverage["total_assets"] > 0]
df_leverage["debt_to_assets"] = df_leverage["long_term_debt"] / df_leverage["total_assets"]

# Verify quartile ranges and median values before plotting
summary_stats = df_leverage.groupby("form_type")["debt_to_assets"].describe(percentiles=[0.25, 0.50, 0.75])
print("Leverage Summary Statistics (Quartiles & Medians):")
print(summary_stats[["count", "mean", "50%", "25%", "75%"]].rename(columns={"50%": "median"}))

#  Plotting Debt-to-Assets Distribution by Form Type
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_leverage,
    x="form_type",
    y="debt_to_assets",
    hue="form_type",
    palette="Set2",
    legend=False,
    showfliers=False,  # Excludes extreme outliers for a cleaner view of the bulk distribution
)

plt.title("Financial Leverage (Long-Term Debt to Total Assets) by Filing Form Type")
plt.xlabel("Form Type")
plt.ylabel("Long-Term Debt / Total Assets Ratio")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Key Observations

---



### 1. Median Leverage Comparison

* **10-Q Filings:** The median financial leverage ratio is slightly higher, sitting at approximately **0.21** (21% of total assets funded by long-term debt).


* **10-K Filings:** The median financial leverage ratio is slightly lower, positioned at around **0.17** (17% of total assets funded by long-term debt).


* **Takeaway:** Overall median leverage across both filing types remains relatively conservative (under 25%), showing that most companies in the dataset maintain manageable long-term debt burdens relative to asset scale.



---

### 2. Interquartile Range (IQR) & Spread

* **10-Q Filings (Middle 50%):** The IQR stretches from approximately **0.08** ($Q_1$) to **0.47** ($Q_3$), indicating a broader spread of leverage ratios during quarterly reporting periods.


* **10-K Filings (Middle 50%):** The IQR is slightly more compact, ranging from around **0.07** ($Q_1$) to **0.37** ($Q_3$).


* **Takeaway:** Annual filings exhibit a narrower, more consolidated central distribution compared to quarterly filings, which reflect intra-year fluctuations in short-to-medium debt restructuring or asset revaluations.



---

### 3. Maximum Whisker Ranges & Upper Extremes

* **10-Q Upper Whisker:** Reaches up to **~1.04**, showing that some interim quarterly periods capture firms with long-term debt exceeding 100% of reported assets.


* **10-K Upper Whisker:** Tops out at approximately **0.81** (excluding fliers).


* **Takeaway:** Quarterly reports capture higher upper-tier debt ratios before annual audit adjustments, debt retirements, or fiscal year-end balance sheet cleanup occur.



---

### 4. Methodological Note

* **Association, Not Causation:** Higher leverage ratios point to an increased reliance on debt financing relative to total assets, but do not imply causality. Differences in capital intensity (e.g., asset-light software vs. asset-heavy industrials/utilities) heavily drive these leverage levels.



# Interactive Scatter Plot: Operating Income vs. Revenues

In [ ]:
import plotly.graph_objects as go
import plotly.express as px


scatter_df = df[(df["revenues"] > 0) & (df["operating_income"] != 0)]

fig_scatter = px.scatter(
    scatter_df,
    x="revenues",
    y="operating_income",
    color="form_type",
    hover_name="ticker",
    log_x=True,
    log_y=True,
    title="Operating Income vs. Revenues by Filing Form Type",
    labels={
        "revenues": "Revenues (Log Scale - USD)",
        "operating_income": "Operating Income (Log Scale - USD)",
        "form_type": "Filing Form Type"
    },
    template="plotly_white",
)
fig_scatter.show()



## Key Observations

---



*  There is a clear upward trend showing a strong positive relationship between revenues and operating income. As revenues increase (from thousands to tens of billions USD), operating income generally increases proportionally.


*  Both axes are plotted on a logarithmic scale (ranging from $1 to $1T for Revenues and $100 to $10B+ for Operating Income). The data points are concentrated heavily in the **$1M to $10B Revenue range** and the **$100k to $100M Operating Income range**.


* The dataset is almost entirely populated by **10-Q filings** (represented by blue dots).


*  Although **10-K** (represented by light peach dots in the legend) is listed as a category, 10-K filings are virtually invisible or obscured beneath the 10-Q data points.


* **Presence of Outliers:** There are several visible outliers, particularly at lower revenue levels (filings showing around $100k in revenue with operating incomes dropping significantly lower or spiking relative to the average trendline).